In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class LayerNorm(nn.Module):
    def __init__(self,emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self,x:torch.tensor):
        mean = x.mean(dim = -1,keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        x = (x-mean)/(var**0.5 + self.eps)
        x = x*self.scale + self.shift
        return x
    

class GELU(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def forward(self,x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2/torch.pi))
                                         * (x+ 0.044715*torch.pow(x,3))))
    

class FeedForward(nn.Module):
    def __init__(self,cfg):
        super().__init__()

        self.ffn = nn.Sequential(nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),
                                 GELU(),
                                 nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"]))
        

    def forward(self,x):
        return self.ffn(x)
    


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in,d_out,context_length,dropout,num_heads,qkv_bias = False):
        super().__init__()
        assert(d_out%num_heads == 0)
        self.num_heads = num_heads
        self.d_out = d_out
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_key = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask',
                             torch.triu(torch.ones(context_length,context_length),diagonal=1))
        self.out_proj = nn.Linear(d_out,d_out,bias=qkv_bias)
        

    def forward(self,x):
        b,token_length,d_in = x.shape
        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)

        q_s = Q.view(b,token_length,self.num_heads,self.head_dim) 
        k_s = K.view(b,token_length,self.num_heads,self.head_dim)
        v_s = V.view(b,token_length,self.num_heads,self.head_dim)


        q_s = q_s.transpose(1,2)
        k_s = k_s.transpose(1,2)
        v_s = v_s.transpose(1,2)

        attention_scores = q_s @ k_s.transpose(2,3)
        mask_bool = self.mask[:token_length,:token_length].bool()
        attention_scores.masked_fill_(mask_bool,-torch.inf)

        attn_weights = torch.softmax(attention_scores/k_s.shape[-1]**0.5,dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vecs = (attn_weights @ v_s).transpose(1,2)
        context_vecs = context_vecs.contiguous().view(b,token_length,self.d_out)
        context_vecs = self.out_proj(context_vecs)
        return context_vecs
    
class TransformerBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1 = LayerNorm(cfg['emb_dim'])
        self.norm2 = LayerNorm(cfg['emb_dim'])
        self.mha = MultiHeadAttention(
            d_in=cfg['emb_dim'],
            d_out=cfg['emb_dim'],
            context_length=cfg['context_length'],
            dropout=cfg['mha_drop_rate'],
            num_heads=cfg['n_heads'],
            qkv_bias=cfg['qkv_bias']
        )
        self.FFN = FeedForward(cfg)
        self.dropout = nn.Dropout(cfg['shortcut_drop_rate'])


    def forward(self,x):
        shortcut = x
        x = self.norm1(x)
        x = self.mha(x)
        x = self.dropout(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.FFN(x)
        x = self.dropout(x)
        x = x + shortcut
        return x
    

class GPTModel(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg['vocab_size'],cfg['emb_dim'])
        self.pos_emb = nn.Embedding(cfg['context_length'],cfg['emb_dim'])
        self.dropout = nn.Dropout(cfg['embedding_drop_rate'])

        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg['n_layers'])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg['emb_dim'],cfg['vocab_size'],bias=False)
        

    def forward(self,inp_x):
        batch_size, seq_len = inp_x.shape
        tok = self.tok_emb(inp_x)
        pos = self.pos_emb(torch.arange(seq_len, device=inp_x.device))

        x = tok + pos
        x = self.dropout(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [3]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories", split="train")
ds.save_to_disk("./tinystories_train")

Saving the dataset (0/4 shards):   0%|          | 0/2119719 [00:00<?, ? examples/s]

In [4]:
# import numpy as np
# import tiktoken
# from datasets import load_from_disk
# from tqdm import tqdm


# ds = load_from_disk("./tinystories_train")
# tok = tiktoken.get_encoding("gpt2")
# EOT = tok.eot_token

# arrays = []
# for ex in tqdm(ds, desc="tokenizing"):
#     ids = tok.encode_ordinary(ex["text"])
#     ids.append(EOT)
#     arrays.append(np.array(ids, dtype=np.uint16))

# num_tokens = sum(len(a) for a in arrays)
# arr = np.memmap("train.bin", dtype=np.uint16, mode="w+", shape=(num_tokens,))
# idx = 0
# for a in arrays:
#     arr[idx:idx+len(a)] = a
#     idx += len(a)


In [5]:
# arr.flush()

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

In [7]:
# class DatasetGPT(Dataset):
#     def __init__(self, arr, max_length, stride=1):
#         """
#         arr: a numpy memmap (or array) of token ids, dtype=uint16
#         max_length: context window length (e.g. 256)
#         stride: step between consecutive windows (e.g. 128 for 50% overlap,
#                 or max_length for non-overlapping)
#         """
#         super().__init__()
#         self.data = arr
#         self.max_length = max_length
#         self.stride = stride

#     def __len__(self):
#         return (len(self.data) - self.max_length - 1) // self.stride + 1

#     def __getitem__(self, index):
#         idx = index * self.stride
#         x = self.data[idx : idx + self.max_length]
#         y = self.data[idx + 1 : idx + self.max_length + 1]
#         # uint16 → int64 conversion is required for PyTorch embeddings
#         x = torch.from_numpy(x.astype(np.int64))
#         y = torch.from_numpy(y.astype(np.int64))
#         return x, y
    

# def create_dataloader_v2(arr, max_length=256, stride=128, batch_size=4,
#                           num_workers=0, shuffle=True, drop_last=True):
#     dataset = DatasetGPT(arr, max_length, stride)
#     return DataLoader(
#         dataset,
#         batch_size=batch_size,
#         shuffle=shuffle,
#         drop_last=drop_last,
#         num_workers=num_workers,
#         pin_memory=True,    # speeds up CPU→GPU transfer
#     )


# train_loader = create_dataloader_v2(
#     arr = arr[:int(len(arr)*0.9)],
#     max_length=256,
#     stride=128,
#     batch_size=2,
#     num_workers=0,
#     shuffle=True,
# )

# val_loader = create_dataloader_v2(
#     arr=arr[int(len(arr)*0.9):],
#     shuffle=False
# )

# x,y = next(iter(val_loader))
# print("INPUT: ", repr(tok.decode(x[0].tolist())))
# print("TARGET:", repr(tok.decode(y[0].tolist())))

In [8]:
MODEL_CONFIG = {
    "vocab_size": 50257,
    "context_length": 512,
    "emb_dim": 768,
    "n_heads":12,
    "n_layers":12,
    "mha_drop_rate":0.1,
    "embedding_drop_rate":0.1,
    "shortcut_drop_rate":0.1,
    "qkv_bias":False
    }

model = GPTModel(MODEL_CONFIG)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(512, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (mha): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (FFN): FeedForward(
        (ffn): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (mha): MultiHeadAt

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken


class GPTDatasetV1(Dataset):
    def __init__(self,txt,tokenizer,maxlength,stride=1):
        super().__init__()
        token_ids= tokenizer.encode(txt)
        self.input_ids = []
        self.output_ids = []

        for i in range(0,len(token_ids)-maxlength,stride):
            self.input_ids.append(torch.tensor(token_ids[i:i+ maxlength]))
            self.output_ids.append(torch.tensor(token_ids[i+1:i+ maxlength+1]))

    
    def __getitem__(self, index):
        return self.input_ids[index], self.output_ids[index]
    
    def __len__(self):
        return len(self.output_ids)
    



def create_dataloader_v1(txt,max_length=256,stride=128,batch_size=4,num_workers=0,shuffle=True,drop_last=True):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt,tokenizer,max_length,stride)
    dl = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dl


file_path = "the_verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()

train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=MODEL_CONFIG["context_length"],
    stride=MODEL_CONFIG["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)
val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=MODEL_CONFIG["context_length"],
    stride=MODEL_CONFIG["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [10]:
print("Train loader:")
len(train_loader)

Train loader:


4

In [49]:
def calc_loss_batch(input_batch,target_batch,model,device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0,1),target_batch.flatten())

    return loss

def calc_loss_loader(dl,model,device,num_batches=None):
    if len(dl) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(dl)
    else:
        num_batches = min(num_batches,len(dl))
    
    i = 0
    dl = iter(dl)
    loss=0

    for i,(x,y) in enumerate(dl):
        if i < num_batches:
            loss+= calc_loss_batch(x,y,model,device).item()
        else:
            break
        
    return loss/num_batches





In [12]:
print(sum(p.numel() for p in model.parameters()))

162607104


In [13]:
next(iter(val_loader))[1]

tensor([[ 8276,  2073,    11,   290,   523,   340,  1392,   284,   307,  2081,
            13,   764,   764,   764,   843,   339, 13055,   520,  5493,  1231,
          1592,  2259,    26,   290,   673,  9174,   262,  4286,  1871,   607,
          5229,   338,  1243,    13,   764,   764, 22135,   198,   198,  1544,
         45111,  2241,   866,   287,   262,  3211,    12, 16337,  1474,  6164,
            11,  8104,   736,   465,  1182,    11,   290, 47425,   278,   465,
          5101, 11061,   340,    11,  3114,   510,   379,   262,  4286,  2029,
           262, 18205,  1681,    12, 12239,    13,   198,   198,     1,    40,
           588,   284, 14996,   326,   520,  5493,  2241,   561,   423,  1813,
           340,   284,   502,    11,   611,   339,  1549,   587,  1498,   284,
           910,   644,   339,  1807,   326,  1110,   526,   198,   198,  1870,
            11,   287,  3280,   284,   257,  1808,   314,  1234,  2063,    12,
          1326,  3147,  1146,   438,     1, 44140,  

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader,model,device)
    val_loss = calc_loss_loader(val_loader,model,device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 11.015482187271118
Validation loss: 11.02595329284668


In [47]:
def train_model_simple(model,train_loader,val_loader,optimizer,device,num_epochs,
                       eval_freq,eval_iter,start_context,tokenizer):
    train_losses,val_losses,track_tokens_seen = [],[],[]
    global_step,tokens_seen = -1,0


    for epoch in range(num_epochs):
        model.train()

        for idx, (x,y) in enumerate(train_loader):
            optimizer.zero_grad()
            loss = calc_loss_batch(x,y,model,device)
            loss.backward()
            optimizer.step()
            tokens_seen += x.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss,val_loss = evaluate_model(model,train_loader,val_loader,device,eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                f"Train loss {train_loss:.3f}, "
                f"Val loss {val_loss:.3f}"
            )
        
        generate_and_print_sample(                     
            model, tokenizer, device, start_context
        )
    return train_losses, val_losses, track_tokens_seen
                
def evaluate_model(model,train_loader,val_loader,device,eval_iter):
    model.eval()
    train_loss = calc_loss_loader(train_loader,model,device,eval_iter)
    val_loss = calc_loss_loader(val_loader,model,device,eval_iter)

    return train_loss,val_loss

def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
        model=model, idx=encoded,
        max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))     
    model.train()

def generate_text_simple(model,idx,max_new_tokens, context_size):
    # idx = (batch,input_tokens)
    for i in range(max_new_tokens):
        idx_cond = idx[:,-context_size:]
        out = model(idx_cond)
        with torch.no_grad():
            text = out[:,-1,:]
        text_softmax = torch.softmax(text,dim=-1)
        tokens = torch.argmax(text_softmax,dim=-1,keepdim=True)
        idx = torch.cat((idx,tokens),dim=1)

    return idx
    
def text_to_token_ids(text,tokenizer):
    encoded = tokenizer.encode(text,allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)   
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)               
    return tokenizer.decode(flat.tolist())


In [50]:
optimizer = torch.optim.AdamW(model.parameters(),lr=0.0004,weight_decay=0.1)
num_epochs = 10
tok = tiktoken.get_encoding("gpt2")
train_losses,val_losses,tokens_seen = train_model_simple(
    model,train_loader,val_loader,optimizer,device,num_epochs,5,5,"Every effort moves you",
    tokenizer=tok
)

Ep 1 (Step 000000): Train loss 9.196, Val loss 9.432
Every effort moves you,,,,,,,,,,.                                       
Ep 2 (Step 000005): Train loss 7.180, Val loss 8.266
Every effort moves you "" "" " " " "" " " "--". "" " " "--" "", and, and I had to the--""--


KeyboardInterrupt: 

In [39]:
model(torch.tensor(tok.encode("Hello, I am Papa Offei")).unsqueeze(0).to(device))

tensor([[[ 0.4745, -0.7527, -0.7731,  ..., -0.6839,  0.0416,  0.6746],
         [ 0.3851,  0.2510, -0.4268,  ..., -0.8354, -0.4202,  0.3386],
         [-0.0532,  0.5518, -0.0294,  ...,  0.1359,  0.2005, -0.8419],
         ...,
         [ 0.2614, -0.2933,  0.1565,  ..., -0.4059, -0.1155, -1.0484],
         [-0.6982,  0.4916, -0.2827,  ...,  0.5456, -0.1155, -0.5245],
         [ 0.9462,  0.2411, -1.1357,  ..., -0.9462, -0.5787, -0.0502]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>)

In [46]:
x,y = next(iter(train_loader))
calc_loss_batch(x,y,model,device)

tensor(10.9941, device='cuda:0', grad_fn=<NllLossBackward0>)